In [1]:
import os
import json
import pandas as pd
import traceback


In [2]:
from dotenv import load_dotenv
load_dotenv() # taking environment variable from .env

True

In [3]:
KEY = os.getenv("GEMINI_API_KEY")
os.environ["GEMINI_API_KEY"] = KEY

In [47]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini Model 
llm = ChatGoogleGenerativeAI(model= "gemini-2.5-flash",temperature = 0.5)





In [52]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("GROQ_API_KEY")
os.environ["GROQ"] = key


In [53]:
llm2 = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [55]:
#simple Query 
response = llm2.invoke("Explain AI in simple terms")
print(response.content)

**What is AI:**
Artificial Intelligence (AI) is a type of computer science that enables machines to think and learn like humans. It's like a super-smart computer that can understand, reason, and make decisions on its own.

**How AI Works:**
Imagine you're trying to teach a child to recognize different animals. You show them pictures of cats, dogs, and birds, and they start to learn what makes each one unique. AI works in a similar way. It's trained on large amounts of data, like pictures, words, or sounds, and it uses that data to learn patterns and make predictions.

**Types of AI:**
There are a few types of AI:

1. **Narrow AI:** This type of AI is designed to perform a specific task, like recognizing faces or playing chess. It's really good at that one thing, but it can't do anything else.
2. **General AI:** This type of AI is like a human brain. It can learn and understand many different things, and it can apply that knowledge to new situations.
3. **Super AI:** This type of AI is 

In [6]:
import logging
import os
from datetime import datetime

In [6]:
datetime.now().strftime('%m_%d_%Y_%H_%M_%S')

'03_28_2026_16_15_59'

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.chains import SequentialChain 
from langchain_community.callbacks import get_openai_callback
import PyPDF2

In [9]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "option" : {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct":"correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "option" : {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct":"correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "option" : {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct":"correct answer",
    },
}

In [10]:
TEMPLATE = """
Text :{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON

{response_json}

"""

In [11]:
quiz_generation_prompt = PromptTemplate(
    input_variable =["text","number","subject","tone","response_json"],
    template = TEMPLATE
)

In [56]:
quiz_chain =LLMChain(llm = llm2 ,prompt = quiz_generation_prompt,output_key = "quiz",verbose = True)

In [15]:

TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [57]:
quiz_evaluation_prompt = PromptTemplate(input_variables = ["subject","quiz"],template =TEMPLATE2)
review_chain = LLMChain(llm = llm2 , prompt = quiz_evaluation_prompt,output_key = "review",verbose=True)

In [18]:
generate_evaluate_chain = SequentialChain(chains = [quiz_chain,review_chain], input_variables=["text","number","subject","tone","response_json"],
output_variables = ["quiz","review"],verbose = True)

In [19]:
file_path = r"D:\LANGCHAIN\Day-7 MCQ generator\data.txt"

In [20]:
file_path 

'D:\\LANGCHAIN\\Day-7 MCQ generator\\data.txt'

In [21]:
with open(file_path,'r') as file:
    TEXT = file.read()

In [22]:
print(TEXT)

Biology is the scientific study of life.[1][2][3] It is a natural science with a broad scope but has several unifying themes that tie it together as a single, coherent field.[1][2][3] For instance, all organisms are made up of cells that process hereditary information encoded in genes, which can be transmitted to future generations. Another major theme is evolution, which explains the unity and diversity of life.[1][2][3] Energy processing is also important to life as it allows organisms to move, grow, and reproduce.[1][2][3] Finally, all organisms are able to regulate their own internal environments.[1][2][3][4][5]

Biologists are able to study life at multiple levels of organization,[1] from the molecular biology of a cell to the anatomy and physiology of plants and animals, and evolution of populations.[1][6] Hence, there are multiple subdisciplines within biology, each defined by the nature of their research questions and the tools that they use.[7][8][9] Like other scientists, bio

In [ ]:
#Serializethe python dictionary into a JSON-formatted string 
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "option": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "option": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "option": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [25]:
NUMBER= 5
SUBJECT = "biology"
TONE = "simple"

In [58]:
#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text :Biology is the scientific study of life.[1][2][3] It is a natural science with a broad scope but has several unifying themes that tie it together as a single, coherent field.[1][2][3] For instance, all organisms are made up of cells that process hereditary information encoded in genes, which can be transmitted to future generations. Another major theme is evolution, which explains the unity and diversity of life.[1][2][3] Energy processing is also important to life as it allows organisms to move, grow, and reproduce.[1][2][3] Finally, all organisms are able to regulate their own internal environments.[1][2][3][4][5]

Biologists are able to study life at multiple levels of organization,[1] from the molecular biology of a cell to the anatomy and physiology of plants and animals, and evolution of populations.[1][6] Hence, there are multiple subdisciplines within biology, each defin

In [59]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:4060
Prompt Tokens:1034
Completion Tokens:3026
Total Cost:0.0


In [60]:
response

{'text': 'Biology is the scientific study of life.[1][2][3] It is a natural science with a broad scope but has several unifying themes that tie it together as a single, coherent field.[1][2][3] For instance, all organisms are made up of cells that process hereditary information encoded in genes, which can be transmitted to future generations. Another major theme is evolution, which explains the unity and diversity of life.[1][2][3] Energy processing is also important to life as it allows organisms to move, grow, and reproduce.[1][2][3] Finally, all organisms are able to regulate their own internal environments.[1][2][3][4][5]\n\nBiologists are able to study life at multiple levels of organization,[1] from the molecular biology of a cell to the anatomy and physiology of plants and animals, and evolution of populations.[1][6] Hence, there are multiple subdisciplines within biology, each defined by the nature of their research questions and the tools that they use.[7][8][9] Like other sci

In [63]:
quiz=response.get("quiz")
print(quiz)


### RESPONSE_JSON

{"1": {"mcq": "What is biology primarily defined as?", "option": {"a": "The study of rocks", "b": "The scientific study of life", "c": "The study of stars", "d": "The study of machines"}, "correct": "b"}, "2": {"mcq": "Which of these is a unifying theme in biology related to hereditary information?", "option": {"a": "All organisms are made of rocks.", "b": "All organisms process hereditary information encoded in genes.", "c": "All organisms can fly.", "d": "All organisms live in water."}, "correct": "b"}, "3": {"mcq": "What major theme in biology helps explain both the similarities and differences among living things?", "option": {"a": "Chemistry", "b": "Physics", "c": "Evolution", "d": "Geology"}, "correct": "c"}, "4": {"mcq": "At what levels can biologists study life?", "option": {"a": "Only at the molecular level", "b": "Only at the population level", "c": "From molecular biology of a cell to evolution of populations", "d": "Only plants and animals"}, "correct": "

In [65]:
quiz = quiz.replace("### RESPONSE_JSON", "").strip() # since it start with ## RESPONSE 
print(quiz)# ##RESPONSE has been removed as it was creating problem to convert th eoutput json into a python dict

{"1": {"mcq": "What is biology primarily defined as?", "option": {"a": "The study of rocks", "b": "The scientific study of life", "c": "The study of stars", "d": "The study of machines"}, "correct": "b"}, "2": {"mcq": "Which of these is a unifying theme in biology related to hereditary information?", "option": {"a": "All organisms are made of rocks.", "b": "All organisms process hereditary information encoded in genes.", "c": "All organisms can fly.", "d": "All organisms live in water."}, "correct": "b"}, "3": {"mcq": "What major theme in biology helps explain both the similarities and differences among living things?", "option": {"a": "Chemistry", "b": "Physics", "c": "Evolution", "d": "Geology"}, "correct": "c"}, "4": {"mcq": "At what levels can biologists study life?", "option": {"a": "Only at the molecular level", "b": "Only at the population level", "c": "From molecular biology of a cell to evolution of populations", "d": "Only plants and animals"}, "correct": "c"}, "5": {"mcq": "

In [66]:
quiz = json.loads(quiz)

In [68]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["option"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [69]:
quiz_table_data

[{'MCQ': 'What is biology primarily defined as?',
  'Choices': 'a: The study of rocks | b: The scientific study of life | c: The study of stars | d: The study of machines',
  'Correct': 'b'},
 {'MCQ': 'Which of these is a unifying theme in biology related to hereditary information?',
  'Choices': 'a: All organisms are made of rocks. | b: All organisms process hereditary information encoded in genes. | c: All organisms can fly. | d: All organisms live in water.',
  'Correct': 'b'},
 {'MCQ': 'What major theme in biology helps explain both the similarities and differences among living things?',
  'Choices': 'a: Chemistry | b: Physics | c: Evolution | d: Geology',
  'Correct': 'c'},
 {'MCQ': 'At what levels can biologists study life?',
  'Choices': 'a: Only at the molecular level | b: Only at the population level | c: From molecular biology of a cell to evolution of populations | d: Only plants and animals',
  'Correct': 'c'},
 {'MCQ': 'Which of the following are examples of eukaryotic org

In [70]:
quiz=pd.DataFrame(quiz_table_data)

In [71]:
quiz

,MCQ,Choices,Correct
0,What is biology primarily defined as?,a: The study of rocks | b: The scientific stud...,b
1,Which of these is a unifying theme in biology ...,a: All organisms are made of rocks. | b: All o...,b
2,What major theme in biology helps explain both...,a: Chemistry | b: Physics | c: Evolution | d: ...,c
3,At what levels can biologists study life?,a: Only at the molecular level | b: Only at th...,c
4,Which of the following are examples of eukaryo...,"a: Archaea and bacteria | b: Protists, fungi, ...",b


In [72]:

quiz.to_csv("biology.csv",index=False)